# Chapter 4 - Regression and Prediction


**Nama:** Aryo Zaki Abdillah  
**NIM:** 101032300060  
**Kelas:** TK47-02  
**Universitas:** Telkom University

> Reproduksi kode dan pendalaman teoretis dari *Practical Statistics for Data Scientists* (O'Reilly), edisi ke-2.
>
> Struktur topik mengikuti notebook Python chapter 1–4 pada repository resmi:
> https://github.com/gedeck/practical-statistics-for-data-scientists

## Tujuan

Chapter ini membahas regression dan prediction mulai dari simple linear regression
hingga multiple regression, model assessment, model selection, weighted regression,
factor variables, correlated predictors, confounding, interactions, diagnostics,
polynomial regression, splines, dan generalized additive models.

Implementasi dibuat dengan Python dan mengikuti konsep utama notebook chapter 4
pada repository resmi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import linregress
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.pipeline import make_pipeline

np.random.seed(42)

## 1. Simple Linear Regression

Simple linear regression menggunakan satu predictor untuk memodelkan outcome.

In [ ]:
n = 200

lung = pd.DataFrame({
    "Exposure": np.random.uniform(0, 20, n)
})

lung["PEFR"] = (
    350
    + 4.2 * lung["Exposure"]
    + np.random.normal(0, 20, n)
)

lung.head()

In [ ]:
predictors = ["Exposure"]
outcome = "PEFR"

model = LinearRegression()
model.fit(lung[predictors], lung[outcome])

print("Intercept:", model.intercept_)
print("Coefficient:", model.coef_[0])

## 2. The Regression Equation

Bentuk umum model: Y = β0 + β1X + ε. Intercept adalah nilai prediksi saat X = 0, sedangkan coefficient menunjukkan perubahan rata-rata Y untuk kenaikan satu unit X.

In [ ]:
x_line = pd.DataFrame({
    "Exposure": np.linspace(
        lung["Exposure"].min(),
        lung["Exposure"].max(),
        100
    )
})

y_line = model.predict(x_line)

plt.figure(figsize=(7, 4))
plt.scatter(lung["Exposure"], lung["PEFR"], alpha=0.4)
plt.plot(x_line["Exposure"], y_line)
plt.xlabel("Exposure")
plt.ylabel("PEFR")
plt.title("Simple Linear Regression")
plt.show()

## 3. Fitted Values dan Residuals

In [ ]:
fitted = model.predict(lung[predictors])
residuals = lung[outcome] - fitted

print("First fitted values:")
print(fitted[:5])

print("\nFirst residuals:")
print(residuals.head())

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(lung["Exposure"], residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Exposure")
plt.ylabel("Residual")
plt.title("Residuals")
plt.show()

## 4. Multiple Linear Regression

Multiple regression menggunakan beberapa predictor untuk menjelaskan satu outcome.

In [ ]:
house = pd.DataFrame({
    "SqFtTotLiving": np.random.normal(2100, 500, n).clip(600, 5000),
    "SqFtLot": np.random.normal(7000, 2500, n).clip(1000, 20000),
    "Bathrooms": np.random.randint(1, 5, n),
    "Bedrooms": np.random.randint(1, 6, n),
    "BldgGrade": np.random.randint(5, 11, n)
})

house["AdjSalePrice"] = (
    60000
    + 180 * house["SqFtTotLiving"]
    + 2.5 * house["SqFtLot"]
    + 18000 * house["Bathrooms"]
    + 9000 * house["Bedrooms"]
    + 25000 * house["BldgGrade"]
    + np.random.normal(0, 75000, n)
)

predictors = [
    "SqFtTotLiving",
    "SqFtLot",
    "Bathrooms",
    "Bedrooms",
    "BldgGrade"
]

outcome = "AdjSalePrice"

house_lm = LinearRegression()
house_lm.fit(house[predictors], house[outcome])

print("Intercept:", house_lm.intercept_)

for name, coef in zip(predictors, house_lm.coef_):
    print(f"{name}: {coef:.4f}")

## 5. Assessing the Model

RMSE mengukur besar error prediksi dalam satuan outcome. R-squared mengukur proporsi variasi outcome yang dijelaskan model.

In [ ]:
fitted = house_lm.predict(house[predictors])

rmse = np.sqrt(
    mean_squared_error(
        house[outcome],
        fitted
    )
)

r2 = r2_score(
    house[outcome],
    fitted
)

print("RMSE:", rmse)
print("R-squared:", r2)

## 6. Regression dengan statsmodels

statsmodels memberikan ringkasan statistik yang lebih detail.

In [ ]:
X_sm = sm.add_constant(house[predictors])

ols_model = sm.OLS(
    house[outcome],
    X_sm
).fit()

print(ols_model.summary())

## 7. Cross-Validation

In [ ]:
cv_scores = cross_val_score(
    house_lm,
    house[predictors],
    house[outcome],
    cv=5,
    scoring="r2"
)

print("Fold scores:", cv_scores)
print("Mean R-squared:", cv_scores.mean())

## 8. Weighted Regression

Weighted regression memberi kontribusi berbeda pada observasi tertentu.

In [ ]:
house["Year"] = np.random.randint(2005, 2021, n)
house["Weight"] = house["Year"] - 2004

weighted_model = LinearRegression()
weighted_model.fit(
    house[predictors],
    house[outcome],
    sample_weight=house["Weight"]
)

print("Weighted intercept:", weighted_model.intercept_)
print("Weighted coefficients:", weighted_model.coef_)

## 9. Factor Variables dan Dummy Variables

Variabel kategorikal perlu direpresentasikan sebagai indicator/dummy variables.

In [ ]:
house["PropertyType"] = np.random.choice(
    ["House", "Apartment", "Townhouse"],
    size=n
)

dummies = pd.get_dummies(
    house["PropertyType"],
    drop_first=True,
    dtype=int
)

print(dummies.head())

In [ ]:
factor_df = pd.concat(
    [house[predictors], dummies],
    axis=1
)

factor_model = LinearRegression()
factor_model.fit(
    factor_df,
    house[outcome]
)

print("Intercept:", factor_model.intercept_)

for name, coef in zip(
    factor_df.columns,
    factor_model.coef_
):
    print(name, coef)

## 10. Correlated Predictors dan Multicollinearity

Multicollinearity terjadi ketika predictor memiliki korelasi kuat satu sama lain. Kondisi ini menyulitkan interpretasi koefisien dan dapat memperbesar ketidakpastian estimasi.

In [ ]:
corr = house[predictors].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True)
plt.title("Predictor Correlation")
plt.show()

## 11. Confounding Variables

Confounding terjadi ketika variabel lain berkaitan dengan predictor dan outcome sehingga hubungan sederhana antara dua variabel dapat berubah setelah variabel tersebut dikontrol.

## 12. Interactions

Interaction berarti pengaruh satu predictor terhadap outcome bergantung pada nilai predictor lain.

In [ ]:
interaction_model = smf.ols(
    "AdjSalePrice ~ SqFtTotLiving * BldgGrade + SqFtLot + Bathrooms + Bedrooms",
    data=house
).fit()

print(interaction_model.summary())

## 13. Regression Diagnostics

Diagnostics digunakan untuk memeriksa asumsi model dan menemukan observasi yang berpotensi bermasalah.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    ols_model.fittedvalues,
    ols_model.resid,
    alpha=0.5
)

plt.axhline(0, linestyle="--")
plt.xlabel("Fitted Values")
plt.ylabel("Residual")
plt.title("Residual vs Fitted")
plt.show()

## 14. Outliers

Outlier merupakan observasi yang jauh dari pola mayoritas. Outlier dapat memengaruhi koefisien dan performa regression.

In [ ]:
influence = ols_model.get_influence()

studentized = influence.resid_studentized_internal
cooks_d = influence.cooks_distance[0]

diagnostic = pd.DataFrame({
    "studentized_residual": studentized,
    "cooks_distance": cooks_d
})

diagnostic.sort_values(
    "cooks_distance",
    ascending=False
).head()

## 15. Heteroskedasticity

Heteroskedasticity terjadi ketika varians residual berubah pada tingkat fitted value yang berbeda.

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(
    x=ols_model.fittedvalues,
    y=np.abs(ols_model.resid),
    scatter_kws={"alpha": 0.25},
    lowess=True
)

plt.xlabel("Fitted Values")
plt.ylabel("Absolute Residual")
plt.title("Heteroskedasticity Check")
plt.show()

## 16. Polynomial Regression

In [ ]:
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)

poly_X = house[["SqFtTotLiving"]]

poly_model.fit(
    poly_X,
    house[outcome]
)

poly_pred = poly_model.predict(poly_X)

print(
    "Polynomial R-squared:",
    r2_score(house[outcome], poly_pred)
)

## 17. Spline Regression

Spline regression membangun fungsi piecewise yang lebih fleksibel untuk pola non-linear.

In [ ]:
spline_model = make_pipeline(
    SplineTransformer(
        n_knots=6,
        degree=3
    ),
    LinearRegression()
)

spline_model.fit(
    house[["SqFtTotLiving"]],
    house[outcome]
)

spline_pred = spline_model.predict(
    house[["SqFtTotLiving"]]
)

print(
    "Spline R-squared:",
    r2_score(house[outcome], spline_pred)
)

## 18. Generalized Additive Model (GAM)

GAM menggabungkan beberapa fungsi smooth dari predictor untuk menangkap pola non-linear.

In [ ]:
try:
    from pygam import LinearGAM, s

    X_gam = house[
        [
            "SqFtTotLiving",
            "SqFtLot",
            "Bathrooms",
            "Bedrooms",
            "BldgGrade"
        ]
    ].values

    y_gam = house[outcome].values

    gam = LinearGAM(
        s(0, n_splines=10)
        + s(1, n_splines=10)
        + s(2)
        + s(3)
        + s(4)
    )

    gam.fit(X_gam, y_gam)

    print(gam.summary())
except ImportError:
    print("Package pygam belum tersedia. Install dengan: pip install pygam")

## Interpretasi

Regression dapat digunakan untuk menjelaskan hubungan antarvariabel maupun
membuat prediction. Simple regression menggunakan satu predictor, sedangkan
multiple regression menggunakan beberapa predictor. Model perlu dievaluasi
menggunakan metrik dan cross-validation, serta diperiksa melalui diagnostics.

Factor variables, interactions, confounding, multicollinearity, outlier,
influential observation, dan heteroskedasticity perlu diperhatikan karena dapat
mempengaruhi interpretasi maupun stabilitas model.

## Kesimpulan Chapter 4

Regression menyediakan kerangka statistik untuk memodelkan hubungan antara
predictor dan outcome. Linear regression merupakan dasar, sedangkan polynomial,
spline, dan GAM dapat digunakan ketika pola hubungan lebih kompleks.